In [1]:
## 싱행 경로 변경하기
import os

# Get the current working directory
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)

# Change the current working directory
new_directory = r'C:\다운로드'  # Corrected path with raw string
os.chdir(new_directory)

# Verify the new working directory
new_directory_confirmed = os.getcwd()
print("New Working Directory:", new_directory_confirmed)

Current Working Directory: C:\python
New Working Directory: C:\다운로드


In [2]:
#필요한 기능 호출 하기
from pytubefix import YouTube
import subprocess
import re
import sys

In [3]:
#비디오 파일과 오디오 파일 다운 받기

def download (url , download_type):

    # Define the URL of the YouTube video
    video_url = url
    result = ''
    # Create a YouTube object
    yt = YouTube(video_url)

    # Select the highest quality video and audio streams
    video_stream = yt.streams.filter(adaptive=True, file_extension='mp4', type='video').first()
    audio_stream = yt.streams.filter(adaptive=True, file_extension='mp4', type='audio').first()

    # Ensure both streams are found
    if not video_stream or not audio_stream:
        raise Exception("Could not find appropriate video or audio streams.")

    # Download video and audio separately
    if download_type == 'all':
        print("Downloading video...")
        video_stream.download(filename="video.mp4")

        print("Downloading audio...")
        audio_stream.download(filename="audio.mp4")

#         subprocess.run([
#         "ffmpeg", "-i", "video.mp4", "-i", "audio.mp4", "-c:v", "copy", "-c:a", "aac", 
#         "-preset", "ultrafast", "-threads", "4", "output.mp4"
#         ], check=True)

        merge_audio_video("video.mp4" , "audio.mp4")
        
        result = 'output.mp4'

    elif download_type == 'audio':
        print("Downloading audio...")
        audio_stream.download(filename="audio.mp4")
        
        
        result = 'audio,mp4'
        
        
    
    return result      




In [4]:
# moviepy로 하나로 합치기 속도 엄청 느림


# from moviepy.editor import *

# # Define the paths of the audio and video files
# audio_path = "audio.mp4"
# video_path = "video.mp4"

# # Create instances of VideoFileClip and AudioFileClip
# video_clip = VideoFileClip(video_path)
# audio_clip = AudioFileClip(audio_path)

# # Merge the audio and video clips
# video_clip_with_audio = video_clip.set_audio(audio_clip)

# # Write the merged video file to a new file
# video_clip_with_audio.write_videofile("file.mp4")

In [5]:
def convert_mp4_to_mp3(input_file, output_file):
    command = [
        "ffmpeg", 
        "-i", input_file, 
        "-preset", "ultrafast",
        "-acodec", "libmp3lame", 
        "-q:a", "12",
        "-threads", "4",
        output_file
    ]
    
    # 먼저 파일의 총 길이를 얻기 위해 FFmpeg를 한 번 호출합니다.
    probe_command = [
        "ffmpeg", "-i", input_file
    ]
    probe_process = subprocess.Popen(probe_command, stderr=subprocess.PIPE, text=True)
    duration = None
    
    # 파일 길이 추출
    for line in probe_process.stderr:
        match = re.search(r"Duration: (\d{2}):(\d{2}):(\d{2})\.(\d{2})", line)
        if match:
            hours = int(match.group(1))
            minutes = int(match.group(2))
            seconds = int(match.group(3))
            duration = hours * 3600 + minutes * 60 + seconds
            break

    probe_process.stderr.close()
    probe_process.wait()

    if duration is None:
        print("Failed to retrieve the duration of the file.")
        return

    # 변환 시작
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # 진행 상황 추적
    for line in iter(process.stderr.readline, ''):
        # 진행 시간을 추출
        time_match = re.search(r"time=(\d{2}):(\d{2}):(\d{2})\.(\d{2})", line)
        if time_match:
            hours = int(time_match.group(1))
            minutes = int(time_match.group(2))
            seconds = int(time_match.group(3))
            current_time = hours * 3600 + minutes * 60 + seconds

            # 진행률 계산
            progress = (current_time / duration) * 100
            sys.stdout.write(f"\rProgress: {progress:.2f}%")
            sys.stdout.flush()

    process.stdout.close()
    process.stderr.close()
    process.wait()

    print("\nConversion completed.")

In [6]:
def merge_audio_video(video_file , audio_file):
    command = ["ffmpeg", "-i", video_file, "-i", audio_file, "-c:v", "copy", "-c:a", "aac", 
        "-preset", "ultrafast", "-threads", "4", "output.mp4"]
    
    probe_command = [
        "ffmpeg", "-i", video_file
    ]
    probe_process = subprocess.Popen(probe_command, stderr=subprocess.PIPE, text=True)
    duration = None
    
    # 파일 길이 추출
    for line in probe_process.stderr:
        match = re.search(r"Duration: (\d{2}):(\d{2}):(\d{2})\.(\d{2})", line)
        if match:
            hours = int(match.group(1))
            minutes = int(match.group(2))
            seconds = int(match.group(3))
            duration = hours * 3600 + minutes * 60 + seconds
            break

    probe_process.stderr.close()
    probe_process.wait()

    if duration is None:
        print("Failed to retrieve the duration of the file.")
        return

    # 변환 시작
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # 진행 상황 추적
    for line in iter(process.stderr.readline, ''):
        # 진행 시간을 추출
        time_match = re.search(r"time=(\d{2}):(\d{2}):(\d{2})\.(\d{2})", line)
        if time_match:
            hours = int(time_match.group(1))
            minutes = int(time_match.group(2))
            seconds = int(time_match.group(3))
            current_time = hours * 3600 + minutes * 60 + seconds

            # 진행률 계산
            progress = (current_time / duration) * 100
            sys.stdout.write(f"\rProgress: {progress:.2f}%")
            sys.stdout.flush()

    process.stdout.close()
    process.stderr.close()
    process.wait()

    print("\nConversion completed.")

In [7]:
def trim_audio(input_file, output_file, duration="00:17:00"):
    # FFmpeg 명령어 정의
    command = [
        "ffmpeg",
        "-i", input_file,            # 입력 파일
        "-t", duration,              # 잘라낼 시간 (17분)
        "-c", "copy",                # 원본 코덱을 그대로 사용
        output_file                  # 출력 파일
    ]

    # 명령어 실행
    subprocess.run(command)

In [8]:
# trim_audio("audio.mp4", "output_audio.mp4", duration="00:17:00")

In [9]:
download ('https://www.youtube.com/watch?v=RxBmiMZeCBE&list=PLkbzizJk4Ae9L-n9yEquO9T927speoMWd&index=3', 'all')

Progress: 100.00%
Conversion completed.


'output.mp4'

In [10]:
# convert_mp4_to_mp3_2('audio.mp4', 'audio6.mp3')

In [11]:
# download ('https://www.youtube.com/watch?v=C-nlHj4tMYs', 'audio')